# Cincinatti car crashes

## Dataset definition

All reported Cincinnati, Ohio car crashes since 2010.

This dataset includes fatal, injury, and non-injury crashes. In compliance with privacy laws, all Public Safety datasets are anonymized

[Original dataset](https://data.cincinnati-oh.gov/safety/Traffic-Crash-Reports-CPD-/rvmt-pkmq/about_data)

**Data Dictionary:**
![image](Data-Dictionary.png)


### Inspiration
- Find which areas of town are the sources of most crashes?
- Figure out how much weather changes the frequency of crashes.
- Predecir accidentes datas en una comunidad dadas unas condiciones.

In [ ]:
# # Instalar dependencias
# !pip install kagglehub
# !pip install rich
# !pip install pandas


In [ ]:
import os
import kagglehub    
from rich import print
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED=1234


In [ ]:
# configuracion de API de Kaggle
os.environ["KAGGLE_USERNAME"] = "user"
os.environ["KAGGLE_KEY"] = "token"


In [ ]:
# Descarga del dataset
path = kagglehub.dataset_download("steverusso/cincinnati-car-crash-data")
print("Path to dataset files:", path)

print("Descargando el dataset")
dataset = pd.read_csv(path+"/cincinnati_traffic_crash_data__cpd.csv") # Carga datos desde un CSV y devuelve un DataFrame 
# dataset.to_csv("archivo.csv", index=False)

dataset.head()

In [ ]:
# Exploracion de datos

print("Info")
print(dataset.info())
print("Valores nulos")
print(dataset.isnull().sum())

## 1. Limpieza inicial de variables

### 1.1  Variables de ID de registros
- La columna ```Unnamed: 0``` corresponde al número de fila, luego la eliminamos porque no aporta información.
- Las columnas ```INSTANCEID``` y ```LOCALREPORTNO``` se refieren al identificador de una incidencia de accidente y al identificador del informe del accidente, respectivamente. No queremos entrenar sobre valores que se refieran a registros.

Se eliminan estas columnas:

In [ ]:
print(f"Tamaño del dataset original: {dataset.shape}")
dataset = dataset.drop(columns=["Unnamed: 0","INSTANCEID", "LOCALREPORTNO"])
print(f"Tamaño del dataset tras eliminar columnas de ID de registros: {dataset.shape}")

### 1.2  Variables de ubicación redundantes

Las siguientes variables sirven para ubicar geográficamente el lugar del incidente. Todas ellas hacen referencia a la ubicación donde ha ocurrido el accidente, pero lo hacen con distintos niveles de precisión. Es conviene reducir redundancia para quedarse con las que aporten más información.

- ```ADDRESS_X```: dirección del accidente, anonimizada a nivel de bloque.
- ```LATITUDE_X```: coordenada geográfica de latitud.
- ```LONGITUDE_X```: coordenada geográfica de longitud.
- ```CPD_NEIGHBORHOOD```: barrio según la división de la policía (Cincinnati Police Department).
- ```SNA_NEIGHBORHOOD```: barrio estadístico de la ciudad de Cincinnati.
- ```ZIP```: código postal.

In [ ]:
geo_columns = ["ADDRESS_X", "CPD_NEIGHBORHOOD", "ZIP", "SNA_NEIGHBORHOOD"]

# Cantidad de valores únicos
for col in geo_columns:
    print(f"{col}: {dataset[col].nunique()} valores únicos")

# Figura con 2 filas y 2 columnas
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()  # para iterar fácilmente

for i, col in enumerate(geo_columns):
    # Top 20 valores más frecuentes
    top_values = dataset[col].value_counts().head(20)
    
    axes[i].bar(top_values.index.astype(str), top_values.values)
    axes[i].set_title(f"Top 20 valores de {col}", fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel("Frecuencia", fontsize=12)
    axes[i].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

En primer lugar, se debe descartar la columna ```ADDRESS_X``` ya que sus valores han sido modificados y toma demasiados valores categóricos únicos para que merezca la pena hacer un _encoding_ sobre ellos.

A continuación, parece razonable mantener las coordenadas geográficas ya que son valores de tipo _float_, que podrán ser usados con facilidad en el proceso de entrenamiento.

Finalmente, las variables ```CPD_NEIGHBORHOOD```, ```SNA_NEIGHBORHOOD```, ```COMMUNITY_COUNCIL_NEIGHBORHOOD``` y ```ZIP``` representan agrupaciones dadas ciertas coordenadas geográficas. Puesto que todas ellas representan un tipo de información similar, tomamos inicialmente ```SNA_NEIGHBORHOOD``` para los análisis, aunque podría ser interesante repetir los resultados obtenidos para las otras. Se ha escogido esta principalmente porque la ciudad de Cincinatti ya había hehco una agrupación estadística de los barrios, además de que tiene pocas categorías únicas.

In [ ]:
dataset = dataset.drop(columns=["ADDRESS_X","CPD_NEIGHBORHOOD", "COMMUNITY_COUNCIL_NEIGHBORHOOD", "ZIP"])
print(f"Tamaño del dataset tras eliminar columnas de variables de ubicación redundantes: {dataset.shape}")

### 1.3  Variables categóricas idénticas

Se ha observado que las variables ```CRASHSEVERITYID``` y ```CRASHSEVERITY``` representan la misma información, ya que existe una correspondencia uno a uno entre ambas. Se decide eliminar la de ```CRASHSEVERITYID``` porque se empleará la otra para el mapa de valores de severidad que se creará más adelante. Realmente es indiferente cuál de las dos eliminar, pero se conserva ```CRASHSEVERITY``` por ser más clara para esta priemra fase.

In [ ]:
# identificar relación uno a uno de los valores
print(dataset.groupby("CRASHSEVERITY")["CRASHSEVERITYID"].unique())
print(dataset.groupby("CRASHSEVERITY")["CRASHSEVERITYID"].nunique())

dataset = dataset.drop(columns=["CRASHSEVERITYID"])
print(f"Tamaño del dataset tras eliminar la columna CRASHSEVERITYID: {dataset.shape}")

Tras esta limpieza inicial hemos pasado de 27 variables (columnas) a 20, que, en prinicipio, cuentan con información útil para entrenar los modelos sobre sus instancias.

In [ ]:
dataset = dataset.drop_duplicates()
print(f"Tamaño del dataset tras eliminar filas duplicadas: {dataset.shape}")

print("Info actualizada:")
print(dataset.info())
print("Valores nulos actualizado:")
print(dataset.isnull().sum())

dataset.head()

## 2) Transformación de los datos

Una vez seleccionadas todas aquellas columnas que, a priori, serán útiles para el análisis, se procede a la trasnformación del tipo de dato para que puedan ser empleados por los diferentes modelos que se desarrollen.

Se observa que hay gran cantidad de columnas con valores nominales, por lo que será necesario valorar el tipo de transformación según el tipo de valores nominales que tomen esas variables, si admiten orden o no.

En primer lugar, las columnas `CRASHDATE` y `DATECRASHREPORTED` se refieren a fechas, por lo que se transforman al tipo de dato de `datetime` para que los modelos puedan aprovechar esta información temporal.

Luego, las siguientes variables son de tipo nominal y sin posibilidad de establecer un orden: 
- `CRASHLOCATION`:
- `MANNEROFCRASH`:
- `UNITTYPE`:
- `GENDER`:
- `SNA_NEIGHBORHOOD`:
- `TYPEOFPERSON`:

Se empleará la transformación de _One-Hot Encoding_ para poder trabajar con los valores de estas variables.

Finalmente, las siguientes varibles tienen valores nominales pero admiten un orden.
- `AGE`:
- `CRASHSEVERITY`:
- `INJURIES`:
- `LIGHTCONDITIONSPRIMARY`:
- `ROADSURFACE`:
- `WEATHER`:
- `DAYOFWEEK`:
- `ROADCONDITIONSPRIMARY`:
- `ROADCONTOUR`:

Se procede con un _Ordinal Encoding_ para poder asignar un valor numérico que respete este orden. Para cada variable, se asignará manualmente un valor de manera ascendente a cada categoría.



In [ ]:
# transformar fechas a datetime
dataset["CRASHDATE"] = pd.to_datetime(dataset["CRASHDATE"], format="%m/%d/%Y %I:%M:%S %p")
dataset["DATECRASHREPORTED"] = pd.to_datetime(dataset["DATECRASHREPORTED"], format="%m/%d/%Y %I:%M:%S %p")

In [ ]:
numeric_cols = [
    "AGE",
    "CRASHSEVERITY",
    "INJURIES",
    "LIGHTCONDITIONSPRIMARY",
    "ROADSURFACE",
    "WEATHER",
    "DAYOFWEEK",
    "ROADCONDITIONSPRIMARY",
    "ROADCONTOUR"
]

OneHotEncoding_cols = [
    "CRASHLOCATION", 
    "MANNEROFCRASH",
    "UNITTYPE",
    "GENDER",
    "SNA_NEIGHBORHOOD",
    "TYPEOFPERSON"
]


# fig, axes = plt.subplots(5, 2, figsize=(20, 15))
# axes = axes.flatten()

# for i, col in enumerate(numeric_cols):
#     dataset[col].value_counts().plot(
#         kind="pie",
#         ax=axes[i],
#         autopct="%1.1f%%"
#     )
#     axes[i].set_title(col)
#     axes[i].set_ylabel("")

# # eliminar subplots vacíos
# for j in range(i+1, len(axes)):
#     fig.delaxes(axes[j])

# plt.suptitle("Distribución CATEGÓRICA", fontsize=16)
# plt.tight_layout()
# plt.show()

# fig, axes = plt.subplots(2, 2, figsize=(20, 15))
# axes = axes.flatten()

# for i, col in enumerate(OneHotEncoding_cols):
#     dataset[col].value_counts().plot(
#         kind="pie",
#         ax=axes[i],
#         autopct="%1.1f%%"
#     )
#     axes[i].set_title(col)
#     axes[i].set_ylabel("")

# # eliminar subplots vacíos
# for j in range(i+1, len(axes)):
#     fig.delaxes(axes[j])

# plt.suptitle("Distribución CATEGÓRICA", fontsize=16)
# plt.tight_layout()
# plt.show()

In [ ]:
# variables nominales con orden
AGE_map = {
    'UNDER 18': 0,
    '18-25': 1,
    '26-30': 2,
    '31-40': 3,
    '41-50': 4,
    '51-60': 5,
    '61-70': 6,
    'OVER 70': 7,
    'UNKNOWN': np.nan
}

CRASHSEVERITY_map = {
    '5 - PROPERTY DAMAGE ONLY': 0,
    '3 - PROPERTY DAMAGE ONLY (PDO)': 0,
    '4 - INJURY POSSIBLE': 1,
    '3 - MINOR INJURY SUSPECTED': 2,
    '2 - INJURY': 3,
    '2 - SERIOUS INJURY SUSPECTED': 4,
    '1 - FATAL INJURY': 5,
    '1 - FATAL': 6
}

INJURIES_map = {
    '1 - NO INJURY / NONE REPORTED': 0,
    '5 - NO APPARENTY INJURY': 1,
    '4 - POSSIBLE INJURY': 2,
    '2 - POSSIBLE': 3,
    '3 - NON-INCAPACITATING': 4,
    '3 - SUSPECTED MINOR INJURY': 5,
    '4 - INCAPACITATING': 6,
    '2 - SUSPECTED SERIOUS INJURY': 7,
    '5 - FATAL': 8,
    '1 - FATAL': 8
}

LIGHTCONDITIONSPRIMARY_map = {
    '1 - DAYLIGHT': 0,
    '2 - DAWN': 1,
    '2 - DUSK': 2,
    '3 - DUSK': 2,
    '3 - DARK - LIGHTED ROADWAY': 3,
    '4 - DARK - LIGHTED ROADWAY': 3,
    '4 - DARK – ROADWAY NOT LIGHTED': 4,
    '5 - DARK – ROADWAY NOT LIGHTED': 4,
    '5 - DARK – UNKNOWN ROADWAY LIGHTING': 5,
    '6 - DARK – UNKNOWN ROADWAY LIGHTING': 5,
    '8 - OTHER': np.nan,
    '9 - OTHER': np.nan,
    '9 - UNKNOWN': np.nan
}

ROADSURFACE_map = {
    '1 - CONCRETE': 0,
    '2 - BLACKTOP, BITUMINOUS, ASPHALT': 1,
    '3 - BRICK/BLOCK': 2,
    '4 - SLAG, GRAVEL, STONE': 3,
    '5 - DIRT': 4,
    '6 - OTHER': np.nan,
    '9 - OTHER': np.nan,
    '9 - UNKNOWN': np.nan
}

WEATHER_map = {
    '1 - CLEAR': 0,
    '2 - CLOUDY': 1,
    '3 - FOG, SMOG, SMOKE': 2,
    '4 - RAIN': 3,
    '5 - SLEET, HAIL': 4,
    '5 - SLEET,HAIL': 4,
    '6 - SNOW': 5,
    '7 - SEVERE CROSSWINDS': 6,
    '8 - BLOWING SAND, SOIL, DIRT, SNOW': 7,
    '9 - FREEZING RAIN OR FREEZING DRIZZLE': 8,
    '9 - OTHER/UNKNOWN': np.nan,
    '99 - OTHER/UNKNOWN': np.nan
}

DAYOFWEEK_map = {
    'MON': 0,
    'TUE': 1,
    'WED': 2,
    'THU': 3,
    'FRI': 4,
    'SAT': 5,
    'SUN': 6
}

ROADCONDITIONPRIMARY_map = {
    '01 - DRY': 0,
    '02 - WET': 1,
    '07 - SLUSH': 2,
    '03 - SNOW': 3,
    '04 - ICE': 4,
    '06 - WATER (STANDING, MOVING)': 5,
    '05 - SAND, MUD, DIRT, OIL, GRAVEL': 6,
    '10 - OTHER': np.nan,
    '09 - OTHER': np.nan,
    '09 - UNKNOWN': np.nan,
    '99 - UNKNOWN': np.nan
}

ROADCONTOUR_map = {
    '1 - STRAIGHT LEVEL': 0,
    '2 - STRAIGHT GRADE': 1,
    '3 - CURVE LEVEL': 2,
    '4 - CURVE GRADE': 3,
    '9 - UNKNOWN': np.nan
}

dataset["AGE"] = dataset["AGE"].map(AGE_map)
dataset["CRASHSEVERITY"] = dataset["CRASHSEVERITY"].map(CRASHSEVERITY_map)
dataset["INJURIES"] = dataset["INJURIES"].map(INJURIES_map)
dataset["LIGHTCONDITIONSPRIMARY"] = dataset["LIGHTCONDITIONSPRIMARY"].map(LIGHTCONDITIONSPRIMARY_map)
dataset["ROADSURFACE"] = dataset["ROADSURFACE"].map(ROADSURFACE_map)
dataset["WEATHER"] = dataset["WEATHER"].map(WEATHER_map)
dataset["DAYOFWEEK"] = dataset["DAYOFWEEK"].map(DAYOFWEEK_map)
dataset["ROADCONDITIONSPRIMARY"] = dataset["ROADCONDITIONSPRIMARY"].map(ROADCONDITIONPRIMARY_map)
dataset["ROADCONTOUR"] = dataset["ROADCONTOUR"].map(ROADCONTOUR_map)

In [ ]:
# unificar valores de la columna GENDER
dataset["GENDER"] = dataset["GENDER"].replace({
    "MALE": "M - MALE",
    "FEMALE": "F - FEMALE"
})

# sustituir todos los valores de UNKNOWN por NaN
for col in OneHotEncoding_cols:
    dataset[col] = dataset[col].astype(str).str.strip()
    mask = dataset[col].str.contains("unknown", case=False, na=False)
    dataset.loc[mask, col] = np.nan

# aplicar One Hot Encoding a las columnas categóricas que no admiten orden
dataset = pd.get_dummies(dataset, columns=OneHotEncoding_cols, dummy_na=True)

In [ ]:
# Info dataset tras el procesado de variables categóricas
print("\nValores nulos (NaN) en columnas categoricas:")
print(dataset[numeric_cols].isna().sum())

print("Info")
print(dataset.info())

print("Columnas del dataset:")
print(list(dataset.columns))

## 2) Data imputation

In [ ]:
# eliminar nan (naive)
previous_shape = dataset.shape[0]
print("Tamaño antes de eliminar NaN:", previous_shape)
dataset = dataset.dropna()
new_shape = dataset.shape[0]
print("Tamaño después de eliminar NaN:", new_shape)
print("Filas eliminadas:", previous_shape - new_shape)

## 3) División de los datos

In [ ]:
# 70% trainset, 20% testset, 10% validationset
trainset, testset = train_test_split(dataset, test_size=0.3, train_size=0.7, random_state=SEED, shuffle=True)
testset, validationset = train_test_split(testset, test_size=2/3, train_size=1/3, random_state=SEED, shuffle=True)